In [1]:
from pandas_plink import read_plink
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

import os
import csv
import json
import gc
import h5py

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from phenotype_levels import phenotype_levels_used

2026-03-27 22:23:27.955506: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 22:23:28.382784: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-27 22:23:28.531509: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-27 22:23:28.570286: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-27 22:23:28.886373: I tensorflow/core/platform/cpu_feature_guar

In [2]:
fam_path = "/storage/plzen4-ntis/home/tadsova/rice_data/base_filtered_v0.7_0.8_10kb_1_0.8_50_1.fam"
pheno_path = "phenotypes/all_phenotypes.txt"
prefix = "/storage/plzen4-ntis/home/tadsova/rice_data/base_filtered_v0.7_0.8_10kb_1_0.8_50_1"

In [3]:
def snp_matrix(prefix):
    """
    Reads PLINK files and converts them into a genotype matrix.
    Handles missing values by imputing them with the most frequent allele 
    for each SNP and returns a compressed int8 matrix.
    """
    (bim, fam, G) = read_plink(prefix, verbose=False)

    X = G.compute().T.astype(np.float32) # sample x snp
    missing_count = np.isnan(X).sum()
    print(f"Celkem chybějících genotypů: {missing_count}")

    # nahrazení NaN za modus
    if missing_count > 0:
        modes = stats.mode(X, axis=0, nan_policy='omit', keepdims=True).mode[0]
        modes = np.nan_to_num(modes, nan=0)
        
        inds = np.where(np.isnan(X))
        X[inds] = np.take(modes, inds[1])
    
    X = X.astype(np.int8)
    print("Genotype matrix shape:", X.shape)

    return X

def get_snp_matrix(prefix, cache_path="snp_matrix/snp_matrix.h5"):
    """
    Loads the genotype matrix from an HDF5 cache file if it exists.
    Otherwise, processes the PLINK files and saves the result to cache.
    """
    if os.path.exists(cache_path):
        print("Načítání matice")
        with h5py.File(cache_path, 'r') as f:
            X = f['X'][:]
        print("Genotype matrix shape:", X.shape)
    else:
        X = snp_matrix(prefix)
        with h5py.File(cache_path, 'w') as f:
            f.create_dataset('X', data=X, compression='gzip', compression_opts=4, chunks=True)
    return X

def prepare_phenotype_data(pheno_path, fam_path, phenotype):
    """
    Merges phenotype data with the .fam file.
    Ensures that only samples present in both files and having a non-null 
    phenotype value are kept.
    """
    pheno = pd.read_csv(pheno_path, sep="\t")
    fam = pd.read_csv(
        fam_path, 
        sep=r"\s+", 
        header=None, 
        names=["FID", "IID", "father", "mother", "sex", "trait"]
    )
    
    pheno_subset = pheno[["3K_DNA_IRIS_UNIQUE_ID", phenotype]].rename(
        columns={"3K_DNA_IRIS_UNIQUE_ID": "IID", phenotype: "VALUE"}
    )
    
    fam["IDX"] = range(len(fam))

    merged_data = fam[["IID", "IDX"]].merge(pheno_subset, on="IID").dropna(subset=["VALUE"])

    merged_data = merged_data[["IDX", "IID", "VALUE"]]
    merged_data["VALUE"] = merged_data["VALUE"].astype(int)

    print(f"Počet vzorků '{phenotype}' po filtraci: {len(merged_data)}")
    
    return merged_data

def analyze_phenotype_distribution(df, phenotype_name):
    """
    Calculates and prints the frequency of each phenotype class.
    Returns the apriori accuracy.
    """
    print(f"\nDistribuce hodnot pro: {phenotype_name}")

    counts = df["VALUE"].value_counts().sort_index()
    print(counts)
    
    total = len(df)
    majority_count = counts.max()
    apriori_acc = majority_count / total
    
    print(f"Apriori accuracy (majority class): {apriori_acc:.4f}")
    
    return apriori_acc

def onehot_encode_phenotype(df, levels, prefix):
    """
    Performs one-hot encoding for the phenotype values based on predefined levels.
    """
    onehot_data = pd.get_dummies(df["VALUE"], prefix=prefix, dtype=np.float32)

    for lvl in levels:
        col = f"{prefix}_{lvl}"
        if col not in onehot_data.columns:
            onehot_data[col] = 0

    column_order = [f"{prefix}_{l}" for l in levels]
    onehot_data = onehot_data[column_order]

    final_df = pd.concat([
        df[["IDX", "IID"]].reset_index(drop=True), 
        onehot_data.reset_index(drop=True)
    ], axis=1)

    print(f'\nOne-hot kódování provedeno.')

    return final_df

def get_clean_labels(df_onehot, levels, prefix):
    """
    Extracts the one-hot encoded values into a clean NumPy array for model training.
    """
    label_cols = []
    for l in levels:
        label_cols.append(prefix + "_" + str(l))
    y = df_onehot[label_cols].values

    return y

def align_genotypes_to_phenotypes(X, df_pheno):
    """
    Filters the full genotype matrix to only include samples that 
    passed phenotype filtering, maintaining the correct order.
    """
    keep_indices = df_pheno["IDX"].values

    X_filtered = X[keep_indices, :]

    return X_filtered

def run_phenotype_pipeline(phenotype_name, X_full_matrix, pheno_path, fam_path):
    """
    The pipeline preparation for a single phenotype:
    1. Loads and filters data.
    2. Analyzes distribution.
    3. Encodes labels.
    4. Aligns genotypes with filtered phenotypes.
    """
    if phenotype_name not in phenotype_levels_used:
        print(f"Fenotyp {phenotype_name} neexistuje.")
        return None
    
    levels = phenotype_levels_used[phenotype_name]
    
    df_pheno = prepare_phenotype_data(pheno_path, fam_path, phenotype_name)
    
    apriori_acc = analyze_phenotype_distribution(df_pheno, phenotype_name)
    
    df_onehot = onehot_encode_phenotype(df_pheno, levels, phenotype_name)
    #print(df_onehot.dtypes)

    y_final = get_clean_labels(df_onehot, levels, phenotype_name)
    
    X_ready_for_onehot = align_genotypes_to_phenotypes(X_full_matrix, df_pheno)

    return X_ready_for_onehot, y_final, levels, apriori_acc

In [4]:
def select_top_snps(X, n_snps=None, method='entropy'):
    """
    Performs feature selection by ranking SNPs based on their variability or information content.
    Supported methods:
    - 'variance': Selects SNPs with the highest statistical variance.
    - 'entropy': Selects SNPs with the highest entropy (information diversity).
    """
    
    if n_snps is None or n_snps >= X.shape[1]:
        return np.arange(X.shape[1])
    
    X_f = X.astype(np.float32)
    
    if method == 'variance':
        scores = np.var(X_f, axis=0)
    
    elif method == 'entropy':
        scores = np.zeros(X_f.shape[1], dtype=np.float32)
        for val in [0, 1, 2]:
            p = np.mean(X_f == val, axis=0)
            p = np.clip(p, 1e-10, 1)
            scores += -p * np.log2(p)
    
    top_indices = np.argsort(scores)[::-1][:n_snps]
    return top_indices

In [5]:
def create_train_val_split(X, y, val_size=0.2, random_state=42):
    """
    Splits the dataset into training and validation sets.
    """
    X_train, X_val, y_train, y_val = train_test_split(
        X, 
        y, 
        test_size=val_size, 
        random_state=random_state, 
        shuffle=True
    )

    print("Dokončené rozdělení dat")
    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
    
    return X_train, X_val, y_train, y_val

In [6]:
class GenomicDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size=32, mask_prob=0.9, shuffle=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.mask_prob = mask_prob
        self.shuffle = shuffle
        self.indices = np.arange(len(self.X))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.X) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, index):
        #změna
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_indices = self.indices[start:end]
        #batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        
        X_batch = self.X[batch_indices]
        y_batch = self.y[batch_indices]

        num_samples, num_snps = X_batch.shape
        X_out = np.zeros((num_samples, num_snps, 4), dtype=np.float32)
        
        mask = np.random.rand(num_samples, num_snps) < self.mask_prob

        for val in [0, 1, 2]:
            X_out[:, :, val] = (X_batch == val) & (~mask)
        
        X_out[:, :, 3] = mask
        
        return X_out.reshape(num_samples, -1), y_batch

In [7]:
def build_genomic_model(input_dim, num_classes, learning_rate=1e-6):
    """
    Builds a simple Softmax Single-layer Neural Network.
    Uses zero initialization for weights to ensure a baseline starting point.
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(num_classes, kernel_initializer='zeros', activation='softmax')
    ])
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy']
    )
    
    return model

In [8]:
def plot_history(history):
    """
    Generates plots for Training/Validation Accuracy and Loss over epochs.
    """
    acc = history.history['categorical_accuracy']
    val_acc = history.history['val_categorical_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'bo-', label='Trénovací acc')
    plt.plot(epochs, val_acc, 'ro-', label='Validační acc')
    plt.title('Přesnost trénování a validace')
    plt.xlabel('Epochs')
    plt.ylabel('Acc')
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'bo-', label='Trénovací loss')
    plt.plot(epochs, val_loss, 'ro-', label='Validační loss')
    plt.title('Ztráta trénování a validace')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [9]:
def run_snp_scaling_experiment(X_ready, y_ready, apriori_acc, snp_counts, method, n_repeats, num_epochs, csv_path, learning_rate):
    """
    Automates a full scaling experiment:
    1. Ranks all SNPs.
    2. Iterates through different SNP subset sizes (snp_counts).
    3. Repeats training multiple times to ensure statistical significance.
    4. Logs results (best epoch metrics) incrementally to a CSV file.
    """

    all_indices = select_top_snps(X_ready, n_snps=None, method=method)
    
    csv_columns = ['run', 'n_snps', 'apriori_acc', 'train_acc', 'val_acc', 'train_loss', 'val_loss', 'epochs_run']
    if not os.path.exists(csv_path):
        pd.DataFrame(columns=csv_columns).to_csv(csv_path, index=False)
        print(f"Vytvořen soubor: {csv_path}")
        existing_results = pd.DataFrame(columns=csv_columns)
    else:
        print(f"Přidávám do souboru: {csv_path}")
        existing_results = pd.read_csv(csv_path)
    
    for n in snp_counts:
        label = n if n is not None else X_ready.shape[1]

        idx = all_indices if n is None else all_indices[:n]
        X_sub = X_ready[:, idx]

        if not existing_results.empty and label in existing_results['n_snps'].values:
            last_run = existing_results[existing_results['n_snps'] == label]['run'].max()
        else:
            last_run = 0

        if last_run >= n_repeats:
            print(f"SNP {label}: hotovo ({last_run}/{n_repeats}), skip")
            continue
        
        for run in range(last_run + 1, n_repeats + 1):
            print(f"\n{'='*50}")
            print(f"SNP: {label} | Run {run}/{n_repeats}")
            
            X_train, X_val, y_train, y_val = create_train_val_split(X_sub, y_ready)
            
            print("y_train shape:", y_train.shape)
            print("unique labels:", np.unique(np.argmax(y_train, axis=1)))

            train_gen = GenomicDataGenerator(X_train, y_train, mask_prob=train_mask_probability)
            val_gen = GenomicDataGenerator(X_val, y_val, mask_prob=test_mask_probability, shuffle=False)
            
            model = build_genomic_model(X_sub.shape[1] * 4, y_train.shape[1], learning_rate=learning_rate)
            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            
            history = model.fit(
                train_gen,
                validation_data=val_gen,
                epochs=num_epochs,
                callbacks=[early_stop],
                #class_weight=class_weight_dict,
                verbose=1
            )

            #plot_history(history)

            best_idx = history.history['val_loss'].index(min(history.history['val_loss']))
            best_val_acc = history.history['val_categorical_accuracy'][best_idx]
            best_val_loss = history.history['val_loss'][best_idx]
            best_train_acc = history.history['categorical_accuracy'][best_idx]
            best_train_loss = history.history['loss'][best_idx]
            
            print(f"  Train acc: {best_train_acc:.4f} | Val acc: {best_val_acc:.4f} | Apriori: {apriori_acc:.4f}")
            
            row = pd.DataFrame([{
                'run': run,
                'n_snps': label,
                'apriori_acc': apriori_acc,
                'train_acc': best_train_acc,
                'val_acc': best_val_acc,
                'train_loss': best_train_loss,
                'val_loss': best_val_loss,
                'epochs_run': len(history.history['loss'])
            }])
            row.to_csv(csv_path, mode='a', header=False, index=False)
    
    results_df = pd.read_csv(csv_path)
    return results_df

In [10]:
# výběr fenotypu
pheno = "PTH"
#pheno = "LPCO_REV_POST"
#pheno = "LSEN"
#pheno = "PEX_REPRO"
#pheno = "APCO_REV_REPRO"

# masky
train_mask_probability = 0.0
test_mask_probability = 0.0

In [11]:
X = get_snp_matrix(prefix)

Načítání matice
Genotype matrix shape: (3024, 404388)


In [12]:
X_ready, y_ready, current_levels, apriori_acc = run_phenotype_pipeline(pheno, X, pheno_path, fam_path)

Počet vzorků 'PTH' po filtraci: 2261

Distribuce hodnot pro: PTH
VALUE
1    348
2    969
3    944
Name: count, dtype: int64
Apriori accuracy (majority class): 0.4286

One-hot kódování provedeno.


In [13]:
# def inspect_top_snps(X, snp_names=None, n_top=10, method='entropy'):
#     """
#     Zobrazí top n_snps podle skóre (entropy nebo variance).
#     """
#     X_f = X.astype(np.float32)

#     if method == 'variance':
#         scores = np.var(X_f, axis=0)
#     elif method == 'entropy':
#         scores = np.zeros(X_f.shape[1], dtype=np.float32)
#         for val in [0, 1, 2]:
#             p = np.mean(X_f == val, axis=0)
#             p = np.clip(p, 1e-10, 1)
#             scores += -p * np.log2(p)

#     top_indices = np.argsort(scores)[::-1][:n_top]

#     rows = []
#     for rank, idx in enumerate(top_indices, 1):
#         counts = {v: int(np.sum(X_f[:, idx] == v)) for v in [0, 1, 2]}
#         total  = X_f.shape[0]
#         rows.append({
#             'rank':    rank,
#             'snp_idx': idx,
#             'name':    snp_names[idx] if snp_names is not None else f'SNP_{idx}',
#             'score':   round(float(scores[idx]), 6),
#             'freq_0':  round(counts[0] / total, 4),
#             'freq_1':  round(counts[1] / total, 4),
#             'freq_2':  round(counts[2] / total, 4),
#             'count_0': counts[0],
#             'count_1': counts[1],
#             'count_2': counts[2],
#         })

#     df = pd.DataFrame(rows).set_index('rank')
#     return df

# df_top = inspect_top_snps(X_ready, snp_names=None, n_top=10, method='entropy')
# print(df_top.to_string())

In [14]:
pth_lr = [1e-5, 1e-6, 1e-7, 1e-8]
snp_counts = [10, 100, 1000, 10000, 100000, None]
top_method = "entropy"
num_repeats = 10
num_epochs = 50

for lr in pth_lr:
    results = run_snp_scaling_experiment(
        X_ready, y_ready,
        apriori_acc=apriori_acc,
        snp_counts=snp_counts,
        method=top_method,
        n_repeats=num_repeats,
        num_epochs=num_epochs,
        learning_rate=lr,
        csv_path=f"PTH_reduction/test_3/snp_LR{lr}_mask.csv"
    )

Vytvořen soubor: PTH_reduction/test_3/snp_LR1e-05_mask.csv

SNP: 10 | Run 1/10
Dokončené rozdělení dat
X_train: (1808, 10), y_train: (1808, 3)
X_val:   (453, 10),   y_val:   (453, 3)
y_train shape: (1808, 3)
unique labels: [0 1 2]
Epoch 1/50


2026-03-27 22:23:39.013768: I tensorflow/core/common_runtime/gpu/gpu_process_state.cc:198] Using CUDA malloc Async allocator for GPU: 0
2026-03-27 22:23:39.015904: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43223 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:82:00.0, compute capability: 8.9
I0000 00:00:1774646619.752500  190511 service.cc:146] XLA service 0x145c18a0a680 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774646619.752535  190511 service.cc:154]   StreamExecutor device (0): NVIDIA L40S, Compute Capability 8.9
2026-03-27 22:23:39.787010: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-27 22:23:39.865016: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90600
'+ptx85' is not a reco

44/56 [======================>.......] - ETA: 0s - loss: 1.0980 - categorical_accuracy: 0.4048

I0000 00:00:1774646620.037105  190511 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


56/56 [==============================] - 1s 6ms/step - loss: 1.0977 - categorical_accuracy: 0.4241 - val_loss: 1.0976 - val_categorical_accuracy: 0.3839
Epoch 2/50
56/56 [==============================] - 0s 3ms/step - loss: 1.0960 - categorical_accuracy: 0.4180 - val_loss: 1.0966 - val_categorical_accuracy: 0.3839
Epoch 3/50
56/56 [==============================] - 0s 4ms/step - loss: 1.0944 - categorical_accuracy: 0.4213 - val_loss: 1.0957 - val_categorical_accuracy: 0.3839
Epoch 4/50
56/56 [==============================] - 0s 3ms/step - loss: 1.0927 - categorical_accuracy: 0.4196 - val_loss: 1.0947 - val_categorical_accuracy: 0.3839
Epoch 5/50
56/56 [==============================] - 0s 3ms/step - loss: 1.0911 - categorical_accuracy: 0.4118 - val_loss: 1.0939 - val_categorical_accuracy: 0.3839
Epoch 6/50
56/56 [==============================] - 0s 3ms/step - loss: 1.0895 - categorical_accuracy: 0.4185 - val_loss: 1.0931 - val_categorical_accuracy: 0.3839
Epoch 7/50
56/56 [=========

...

56/56 [==============================] - 13s 228ms/step - loss: 1.0736 - categorical_accuracy: 0.4129 - val_loss: 1.0713 - val_categorical_accuracy: 0.3839
Epoch 2/50
56/56 [==============================] - 13s 228ms/step - loss: 1.0401 - categorical_accuracy: 0.4263 - val_loss: 1.0550 - val_categorical_accuracy: 0.4308
Epoch 3/50
56/56 [==============================] - 13s 230ms/step - loss: 1.0248 - categorical_accuracy: 0.4163 - val_loss: 1.0460 - val_categorical_accuracy: 0.4263
Epoch 4/50
56/56 [==============================] - 13s 234ms/step - loss: 1.0184 - categorical_accuracy: 0.4263 - val_loss: 1.0397 - val_categorical_accuracy: 0.4286
Epoch 5/50
56/56 [==============================] - 12s 212ms/step - loss: 1.0168 - categorical_accuracy: 0.4252 - val_loss: 1.0360 - val_categorical_accuracy: 0.4442
Epoch 6/50
56/56 [==============================] - 13s 232ms/step - loss: 1.0143 - categorical_accuracy: 0.4152 - val_loss: 1.0330 - val_categorical_accuracy: 0.4777
Epoch 7/5

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


56/56 [==============================] - 14s 239ms/step - loss: 1.0716 - categorical_accuracy: 0.4208 - val_loss: 1.0717 - val_categorical_accuracy: 0.4598
Epoch 2/50
56/56 [==============================] - 13s 231ms/step - loss: 1.0375 - categorical_accuracy: 0.4241 - val_loss: 1.0563 - val_categorical_accuracy: 0.4598
Epoch 3/50
56/56 [==============================] - 13s 240ms/step - loss: 1.0245 - categorical_accuracy: 0.4241 - val_loss: 1.0470 - val_categorical_accuracy: 0.4330
Epoch 4/50
56/56 [==============================] - 13s 224ms/step - loss: 1.0195 - categorical_accuracy: 0.4202 - val_loss: 1.0416 - val_categorical_accuracy: 0.4353
Epoch 5/50
56/56 [==============================] - 12s 210ms/step - loss: 1.0160 - categorical_accuracy: 0.4023 - val_loss: 1.0376 - val_categorical_accuracy: 0.4330
Epoch 6/50
56/56 [==============================] - 13s 238ms/step - loss: 1.0141 - categorical_accuracy: 0.4286 - val_loss: 1.0352 - val_categorical_accuracy: 0.4286
Epoch 7/5

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


56/56 [==============================] - 14s 249ms/step - loss: 1.0738 - categorical_accuracy: 0.4219 - val_loss: 1.0713 - val_categorical_accuracy: 0.4062
Epoch 2/50
56/56 [==============================] - 13s 236ms/step - loss: 1.0404 - categorical_accuracy: 0.4241 - val_loss: 1.0561 - val_categorical_accuracy: 0.4420
Epoch 3/50
56/56 [==============================] - 12s 220ms/step - loss: 1.0260 - categorical_accuracy: 0.4247 - val_loss: 1.0457 - val_categorical_accuracy: 0.4531
Epoch 4/50
56/56 [==============================] - 15s 273ms/step - loss: 1.0178 - categorical_accuracy: 0.4252 - val_loss: 1.0402 - val_categorical_accuracy: 0.4554
Epoch 5/50
56/56 [==============================] - 12s 220ms/step - loss: 1.0164 - categorical_accuracy: 0.4152 - val_loss: 1.0363 - val_categorical_accuracy: 0.4241
Epoch 6/50
56/56 [==============================] - 13s 228ms/step - loss: 1.0145 - categorical_accuracy: 0.4297 - val_loss: 1.0335 - val_categorical_accuracy: 0.4464
Epoch 7/5

In [15]:
"""
learning_rate = 1e-5
snp_counts = [None]
method = "entropy"
num_epochs = 100

all_indices = select_top_snps(X_ready, n_snps=None, method=method)
all_results = []

for n in snp_counts:
    label = n if n is not None else X_ready.shape[1]
    idx = all_indices if n is None else all_indices[:n]
    X_sub = X_ready[:, idx]

    print(f"\n{'='*50}")
    print(f"SNP: {label}")

    X_train, X_val, y_train, y_val = create_train_val_split(X_sub, y_ready)
    y_integers = np.argmax(y_train, axis=1)
    class_weight_dict = dict(enumerate(compute_class_weight(
        class_weight='balanced', classes=np.unique(y_integers), y=y_integers
    )))

    train_gen = GenomicDataGenerator(X_train, y_train, mask_prob=test_mask_probability)
    val_gen = GenomicDataGenerator(X_val, y_val, mask_prob=test_mask_probability, shuffle=False)

    model = build_genomic_model(X_sub.shape[1] * 4, y_train.shape[1], learning_rate=learning_rate)
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=num_epochs,
        callbacks=[early_stop],
        class_weight=class_weight_dict,
        verbose=1
    )

    best_idx = history.history['val_loss'].index(min(history.history['val_loss']))
    best_val_acc = history.history['val_categorical_accuracy'][best_idx]
    best_val_loss = history.history['val_loss'][best_idx]
    best_train_acc = history.history['categorical_accuracy'][best_idx]
    best_train_loss = history.history['loss'][best_idx]

    print(f"  Train acc: {best_train_acc:.4f} | Val acc: {best_val_acc:.4f} | Apriori: {apriori_acc:.4f}")

    all_results.append({
        'n_snps': label,
        'apriori_acc': apriori_acc,
        'train_acc': best_train_acc,
        'val_acc': best_val_acc,
        'train_loss': best_train_loss,
        'val_loss': best_val_loss,
        'epochs_run': len(history.history['loss'])
    })

results_df = pd.DataFrame(all_results)
"""

'\nlearning_rate = 1e-5\nsnp_counts = [None]\nmethod = "entropy"\nnum_epochs = 100\n\nall_indices = select_top_snps(X_ready, n_snps=None, method=method)\nall_results = []\n\nfor n in snp_counts:\n    label = n if n is not None else X_ready.shape[1]\n    idx = all_indices if n is None else all_indices[:n]\n    X_sub = X_ready[:, idx]\n\n    print(f"\n{\'=\'*50}")\n    print(f"SNP: {label}")\n\n    X_train, X_val, y_train, y_val = create_train_val_split(X_sub, y_ready)\n    y_integers = np.argmax(y_train, axis=1)\n    class_weight_dict = dict(enumerate(compute_class_weight(\n        class_weight=\'balanced\', classes=np.unique(y_integers), y=y_integers\n    )))\n\n    train_gen = GenomicDataGenerator(X_train, y_train, mask_prob=test_mask_probability)\n    val_gen = GenomicDataGenerator(X_val, y_val, mask_prob=test_mask_probability, shuffle=False)\n\n    model = build_genomic_model(X_sub.shape[1] * 4, y_train.shape[1], learning_rate=learning_rate)\n    early_stop = EarlyStopping(monitor=\